# 01 — ML | Game Win-Probability Model

**Project:** MLB Performance Intelligence Platform

Predicts the probability that the **home team wins**, from the point-in-time features in
`gold.game_features`.

| Step | Detail |
|---|---|
| Split | **Time-based**: train on seasons before `test_season`, evaluate on `test_season` (never seen in training) |
| Baselines | Always pick the home team; pick the team with the better win % to date |
| Models | Logistic regression (standardized features) and gradient-boosted trees (`HistGradientBoostingClassifier`) |
| Metrics | Accuracy, log loss, Brier score, ROC AUC, calibration by probability bucket |
| Selection | Lowest test **log loss** (rewards honest probabilities, not just picks) |
| Final fit | The selected model is refit on all completed games and scores the upcoming games |
| Explanation | Per game, the three features that move the logistic model's probability the most |
| Tracking | Parameters, metrics and the model are logged to **MLflow** when available |

Outputs: `gold.win_model_metrics`, `gold.win_model_calibration`, `gold.win_model_features`, `gold.game_predictions`.

Probabilities, not certainties: even very good MLB models pick the winner in roughly 55–60 % of games.

In [ ]:
dbutils.widgets.text("test_season", "", "Test season (blank = latest completed season)")

In [ ]:
import sys, importlib, json
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

try:
    import sklearn  # noqa: F401  (preinstalled on Databricks serverless and ML runtimes)
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn"])

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    import mlflow
    import mlflow.sklearn
except Exception:  # local test harness without MLflow
    mlflow = None

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)
FEATURES = ["d_win_pct", "d_run_diff", "d_last10", "d_split", "d_starter_fip", "d_starter_kbb",
            "d_bullpen_era", "d_bullpen_fatigue", "d_rest", "park_factor", "is_postseason"]
LABELS = {
    "d_win_pct": "Team record", "d_run_diff": "Run differential", "d_last10": "Last 10 games",
    "d_split": "Home/road split", "d_starter_fip": "Starting pitcher (FIP)", "d_starter_kbb": "Starter strikeouts−walks",
    "d_bullpen_era": "Bullpen ERA", "d_bullpen_fatigue": "Bullpen rest", "d_rest": "Days of rest",
    "park_factor": "Ballpark", "is_postseason": "Postseason",
}

df = spark.table(T("gold", "game_features")).toPandas()
df["official_date"] = pd.to_datetime(df["official_date"])
done = df[df["home_win"].notna()].copy()
done["home_win"] = done["home_win"].astype(int)
upcoming = df[~df["is_completed"]].copy()

TEST_SEASON = int(dbutils.widgets.get("test_season") or done.loc[done["season_type"] == "Regular Season", "season"].max())
train = done[done["season"] < TEST_SEASON]
test = done[done["season"] == TEST_SEASON]
print(f"Train: {len(train)} games ({sorted(train.season.unique())}) | Test: {len(test)} games ({TEST_SEASON}) | Upcoming: {len(upcoming)}")

## Train, compare and select

In [ ]:
def metrics(name, y, p, split):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {"model": name, "split": split, "games": int(len(y)), "accuracy": float(accuracy_score(y, p >= 0.5)),
            "log_loss": float(log_loss(y, p, labels=[0, 1])), "brier": float(brier_score_loss(y, p)),
            "auc": float(roc_auc_score(y, p)) if len(set(y)) > 1 else None}

home_rate = train["home_win"].mean()
rows = [
    metrics("Baseline: home team", test["home_win"], np.full(len(test), home_rate), "test"),
    metrics("Baseline: better record", test["home_win"],
            np.where(test["d_win_pct"] > 0, 0.6, np.where(test["d_win_pct"] < 0, 0.4, home_rate)), "test"),
]
candidates = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.04, max_iter=250,
                                                        l2_regularization=1.0, min_samples_leaf=60, random_state=7),
}
test_probs = {}
for name, model in candidates.items():
    model.fit(train[FEATURES], train["home_win"])
    rows.append(metrics(name, train["home_win"], model.predict_proba(train[FEATURES])[:, 1], "train"))
    test_probs[name] = model.predict_proba(test[FEATURES])[:, 1]
    rows.append(metrics(name, test["home_win"], test_probs[name], "test"))

metrics_df = pd.DataFrame(rows)
best = min(candidates, key=lambda n: metrics_df.query("model == @n and split == 'test'")["log_loss"].iloc[0])
metrics_df["selected"] = metrics_df["model"] == best
metrics_df["test_season"] = TEST_SEASON
metrics_df["trained_at"] = pd.Timestamp.now("UTC").tz_localize(None)
print(metrics_df[["model", "split", "games", "accuracy", "log_loss", "brier", "auc"]].round(4).to_string(index=False))
print(f"\nSelected: {best}")

## Calibration, feature importance and per-game explanations

In [ ]:
p_best = test_probs[best]
fav_p = np.maximum(p_best, 1 - p_best)
fav_won = np.where(p_best >= 0.5, test["home_win"], 1 - test["home_win"])
edges = [0.5, 0.55, 0.6, 0.65, 0.7, 1.0]
bucket = pd.cut(fav_p, edges, right=False, include_lowest=True, labels=["50–55%", "55–60%", "60–65%", "65–70%", "70%+"])
calib = (pd.DataFrame({"bucket": bucket.astype(str), "predicted": fav_p, "won": fav_won})
         .groupby("bucket", observed=True).agg(games=("won", "size"), predicted=("predicted", "mean"), actual=("won", "mean"))
         .reset_index())
calib["test_season"] = TEST_SEASON
print(calib.round(3).to_string(index=False))

# Logistic model (refit on all completed games) explains each prediction: coefficient × standardized value
explainer = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)).fit(done[FEATURES], done["home_win"])
scaler, lr = explainer.named_steps["standardscaler"], explainer.named_steps["logisticregression"]
coef = pd.Series(lr.coef_[0], index=FEATURES)
importance = pd.DataFrame({"feature": FEATURES, "label": [LABELS[f] for f in FEATURES],
                           "coefficient": coef.values, "abs_importance": np.abs(coef.values)}).sort_values("abs_importance", ascending=False)

def top_factors(frame):
    z = scaler.transform(frame[FEATURES]) * lr.coef_[0]
    out = []
    for row in z:
        idx = np.argsort(-np.abs(row))[:3]
        out.append(json.dumps([{"f": FEATURES[i], "v": round(float(row[i]), 3)} for i in idx]))
    return out

## Final model: refit on all completed games, score upcoming games, write Gold

In [ ]:
final = candidates[best]
final.fit(done[FEATURES], done["home_win"])

def frame(part, probs, split):
    out = part[["game_pk", "season", "season_type", "official_date", "home_team_id", "away_team_id",
                "home_starter_id", "away_starter_id", "home_starter_known", "away_starter_known",
                "home_sp_fip", "away_sp_fip", "home_wp", "away_wp", "home_last10", "away_last10",
                "park_factor", "home_win"]].copy()
    out["official_date"] = out["official_date"].dt.date
    out["home_win_prob"] = probs
    out["predicted_winner_id"] = np.where(probs >= 0.5, out["home_team_id"], out["away_team_id"])
    out["correct"] = np.where(out["home_win"].isna(), None, (probs >= 0.5) == (out["home_win"] == 1))
    out["split"] = split
    out["top_factors"] = top_factors(part)
    return out

pred = pd.concat([
    frame(train, candidates[best].fit(train[FEATURES], train["home_win"]).predict_proba(train[FEATURES])[:, 1], "train"),
    frame(test, p_best, "test"),
    frame(upcoming, final.fit(done[FEATURES], done["home_win"]).predict_proba(upcoming[FEATURES])[:, 1] if len(upcoming) else np.array([]), "upcoming"),
], ignore_index=True)
pred["model"] = best
for c in pred.select_dtypes("float").columns:  # pandas NaN would become Spark NaN, not NULL
    pred[c] = pred[c].astype(object).where(pred[c].notna(), None)
pred["correct"] = pred["correct"].astype("object")

with P.log_step(spark, RUN_ID, "ml", "win_probability_model", f"test_season={TEST_SEASON}") as step:
    P.replace_table(spark, spark.createDataFrame(metrics_df), T("gold", "win_model_metrics"),
        "Gold: win-probability model metrics (train/test, baselines, selected model)")
    P.replace_table(spark, spark.createDataFrame(calib), T("gold", "win_model_calibration"),
        "Gold: calibration of the selected model on the test season (predicted vs actual win rate of the favourite)")
    P.replace_table(spark, spark.createDataFrame(importance), T("gold", "win_model_features"),
        "Gold: standardized logistic-regression coefficients (what moves the win probability)")
    pred_sdf = spark.createDataFrame(pred.assign(correct=pred["correct"].map(lambda v: None if v is None else bool(v))))
    step.records = P.replace_table(spark, pred_sdf, T("gold", "game_predictions"),
        "Gold: home-team win probability per game (train in-sample, test out-of-sample, upcoming scored by the final model)")
    step.message = f"{best} · test accuracy {metrics_df.query('selected and split == \"test\"')['accuracy'].iloc[0]:.3f}"

if mlflow is not None:
    try:
        with mlflow.start_run(run_name=f"win_probability_{TEST_SEASON}"):
            mlflow.log_params({"model": best, "test_season": TEST_SEASON, "features": ",".join(FEATURES)})
            for r in rows:
                if r["model"] == best:
                    mlflow.log_metrics({f"{r['split']}_{k}": v for k, v in r.items() if k in ("accuracy", "log_loss", "brier", "auc") and v is not None})
            mlflow.sklearn.log_model(final, "model", input_example=done[FEATURES].head(5))
    except Exception as e:  # tracking is optional; never fail the job because of it
        print(f"MLflow logging skipped: {e}")

In [ ]:
display(spark.table(T("gold", "win_model_metrics")))
display(spark.sql(f"""SELECT official_date, home_team_id, away_team_id, ROUND(home_win_prob, 3) AS home_win_prob, top_factors
                      FROM {T('gold', 'game_predictions')} WHERE split = 'upcoming' ORDER BY official_date LIMIT 10"""))